In [ ]:
# Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
def parse_conll_file(conll_file_path):
    data = []
    tokens = []
    ner_tags = []
    tag2id = {"O": 0, "Implicature": 1, "Ambiguity": 2, "Presupposition": 3, "Explicit": 4}

    with open(conll_file_path, 'r') as f:
        for i, line in enumerate(f, start=1):
            line = line.strip()
            if line == "":
                if tokens:
                    data.append({'id': str(len(data)), 'ner_tags': ner_tags, 'tokens': tokens})
                    tokens, ner_tags = [], []
            else:
                parts = line.split()
                if len(parts) != 3:
                    continue
                token, _, tag = parts
                tokens.append(token)
                if tag in ["B-Implicature", "I-Implicature"]:
                    ner_tags.append(tag2id["Implicature"])
                elif tag in ["B-Ambiguity", "I-Ambiguity"]:
                    ner_tags.append(tag2id["Ambiguity"])
                elif tag in ["B-Presupposition", "I-Presupposition"]:
                    ner_tags.append(tag2id["Presupposition"])
                elif tag in ["B-Explicit", "I-Explicit"]:
                    ner_tags.append(tag2id["Explicit"])
                elif tag == "O":
                    ner_tags.append(tag2id["O"])

        if tokens:
            data.append({'id': str(len(data)), 'ner_tags': ner_tags, 'tokens': tokens})

    return data

In [ ]:
conll_file_path = "/path/to/your/file"
data = parse_conll_file(conll_file_path)

# Print the formatted data
for entry in data:
    print(entry)

In [ ]:
print(len(data))

In [ ]:
!pip install transformers
!pip install accelerate -U
!pip install datasets

In [ ]:
!pip install bitsandbytes accelerate transformers

In [ ]:
!pip install --upgrade transformers accelerate bitsandbytes

In [ ]:
# Mapping for conversion
id2label = {0: "O", 1: "Implicature", 2: "Ambiguity", 3: "Presupposition", 4: "Explicit"}
label2id = {"O": 0, "Implicature": 1, "Ambiguity": 2, "Presupposition": 3, "Explicit": 4}

In [ ]:
# A function to attach punctuation to sentences
import re

def reconstruct_sentence(tokens):
    sentence = ""
    for i, token in enumerate(tokens):
        if re.match(r"^[.,!?;:’”')\]]$", token):
            sentence += token
        elif token in ['(', '[', '“', '‘']:
            if sentence and not sentence.endswith(" "):
                sentence += " "
            sentence += token
        else:
            if sentence and not sentence.endswith(" "):
                sentence += " "
            sentence += token
    return sentence

In [ ]:
# Creating a df with sentences instead of separate tokens
import pandas as pd

records = []
for sent in data:
    sentence_txt = reconstruct_sentence(sent["tokens"])
    label_txt = " ".join(id2label[t] for t in sent["ner_tags"])
    records.append({"sentence": sentence_txt, "ner_tag": label_txt})

df = pd.DataFrame(records)

In [ ]:
print(df.head(5))

In [ ]:
from huggingface_hub import login

# Make sure you are logged in to Hugging Face
# You can authenticate by running: huggingface-cli login
# Or by passing your token: login(token="YOUR_TOKEN")
login()

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig

In [ ]:
def build_prompt(sentence):
    prompt = f"""Your task is to analyze each sentence of the text and determine whether it is *Explicit*, *Implicature*, *Ambiguity*, *Presupposition*.
Explicit refers to transparent and clearly understandable content.
Implicature refers to content that reflects universal knowledge, or that can be inferred from the utterance and its context, though not explicitly stated.
Ambiguity refers to content that may denote multiple entities or states of affairs and cannot be resolved from context alone, requiring further precision.
Presupposition refers to presenting a notion as already shared by the author and their addressee(s). Presuppositions convey information openly, assuming that it is already known and accepted.
Implicature, Ambiguity, Presupposition are subcategories of Implicit (hidden meaning).
The text surrounding the sentence is your context.

Instructions:
- For each Explicit, Implicature, Ambiguity, Presupposition found, return a separate JSON object with exactly two fields:
  - "sentence": the exact span from the input sentence expressing the argument.
  - "type": "Explicit" or "Implicature" or "Ambiguity", "Presupposition".
- If none of them is found, return one JSON object with both fields set to "".
- Do **not** wrap the JSON objects in a list (no square brackets).
- Separate multiple JSON objects with **commas and spaces only**, e.g.:
  {{ "sentence": "...", "type": "Implicature" }}, {{ "sentence": "...", "type": "Presupposition" }}
- The output must be strictly valid JSON:
  - Use double quotes only
  - Close all braces correctly
  - Do not include trailing commas
- Do not include any explanation, notes, or extra text. Output **only** the JSON objects.

Sentence:
{sentence}
"""
    return prompt

In [ ]:
# Load the model
model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="float16"
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", quantization_config=bnb_config)

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=2048, do_sample=False)

In [ ]:
predictions = []

for row in tqdm(df.itertuples(), total=len(df)):
    prompt = build_prompt(row.sentence)
    output = pipe(prompt)[0]['generated_text']

    output_clean = output.replace(prompt, "").strip()

    predictions.append({
        "sentence": row.sentence,
        "gold": row.ner_tag,
        "mistral_output": output_clean
    })

In [ ]:
import json

In [ ]:
output_path = "/path/to/your/project/finegrained_predictions.json"
with open(output_path, "w") as f:
    json.dump(predictions, f, indent=2)

print(f"Saved predictions to {output_path}")

In [ ]:
with open("/path/to/your/project/finegrained_predictions.json") as f:
    predictions = json.load(f)

In [ ]:
import json
import nltk
from difflib import SequenceMatcher
from sklearn.metrics import classification_report
from nltk.tokenize import word_tokenize, sent_tokenize

nltk.download('punkt_tab')

In [ ]:
print(predictions[1]["sentence"])
print(predictions[1]["gold"])
print(predictions[1]["mistral_output"])

In [ ]:
import re

def extract_mistral_sentences(mistral_output):
    mistral_output = mistral_output.strip()

    # Remove the "Output:" prefix if present
    if mistral_output.lower().startswith("output:"):
        mistral_output = mistral_output[len("output:"):].strip()

    # Extract JSON-like pairs: {"sentence": "...", "type": "..."}
    pattern = r'\{\s*"sentence"\s*:\s*"(.+?)"\s*,\s*"type"\s*:\s*"(.+?)"\s*\}'
    matches = re.findall(pattern, mistral_output, re.DOTALL)

    return [(s.strip(), t.strip().lower()) for s, t in matches]

def get_sentence_level_labels(text, gold_labels):
    tokens = word_tokenize(text.strip())
    labels = gold_labels.strip().split()

    if len(tokens) != len(labels):
        print(f"⚠️ Warning: token-label mismatch: {len(tokens)} tokens vs {len(labels)} labels")

    sentences = sent_tokenize(text.strip())
    sentence_labels = []

    idx = 0
    for sent in sentences:
        sent_len = len(word_tokenize(sent))
        sent_labels = labels[idx:idx + sent_len]

        if not sent_labels:
            sentence_labels.append((sent.strip(), "O"))
        elif any("Implicature" in tag for tag in sent_labels):
            sentence_label = "Implicature"
        elif any("Ambiguity" in tag for tag in sent_labels):
            sentence_label = "Ambiguity"
        elif any("Presupposition" in tag for tag in sent_labels):
            sentence_label = "Presupposition"
        elif any("Explicit" in tag for tag in sent_labels):
            sentence_label = "Explicit"
        else:
            sentence_label = "O"

        sentence_labels.append((sent.strip(), sentence_label))
        idx += sent_len

    return sentence_labels

def best_match(pred_sent, gold_sents):
    pred_clean = pred_sent.lower()
    best_idx, best_score = -1, 0
    for i, (gold_sent, _) in enumerate(gold_sents):
        score = SequenceMatcher(None, pred_clean, gold_sent.lower()).ratio()
        if score > best_score:
            best_idx, best_score = i, score
    return best_idx if best_score > 0.8 else -1  # threshold

gold_all = []
pred_all = []

for item in predictions:
    gold_pairs = get_sentence_level_labels(item["sentence"], item["gold"])
    pred_pairs = extract_mistral_sentences(item["mistral_output"])

    matched = set()
    pred_sent_map = {}

    for pred_sent, pred_label in pred_pairs:
        idx = best_match(pred_sent, gold_pairs)
        if idx != -1 and idx not in matched:
            pred_sent_map[idx] = pred_label
            matched.add(idx)

    for i, (gold_sent, gold_label) in enumerate(gold_pairs):
      pred_label = pred_sent_map.get(i, "O")  # assign "O" if not predicted

      # Normalize both labels to have capitalized first letter
      gold_label = gold_label.strip().capitalize()
      pred_label = pred_label.strip().capitalize()

      gold_all.append(gold_label)
      pred_all.append(pred_label)

print(classification_report(
    gold_all,
    pred_all,
    labels=["Implicature", "Ambiguity", "Presupposition", "Explicit", "O"],
    digits=4
))